# Ingenieria de Variables — Deteccion de Fraude Federado

## Objetivo
Minimizar falsos positivos en compras presenciales: reducir las alertas falsas en transacciones realizadas fisicamente con la tarjeta presente (canales POS y ATM).

In [1]:
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)

bolivia   = pd.read_csv('../data/bolivia_dataset.csv',   sep=';')
brasil    = pd.read_csv('../data/brasil_dataset.csv',    sep=';')
guatemala = pd.read_csv('../data/guatemala_dataset.csv', sep=';')

print('Bolivia:',   bolivia.shape,   '| fraudes:', bolivia['is_fraud'].sum())
print('Brasil:',    brasil.shape,    '| fraudes:', brasil['is_fraud'].sum())
print('Guatemala:', guatemala.shape, '| fraudes:', guatemala['is_fraud'].sum())

Bolivia: (100003, 66) | fraudes: 4919
Brasil: (100000, 66) | fraudes: 3205
Guatemala: (100000, 66) | fraudes: 0.0


## Eliminacion de columnas inutiles o peligrosas

Se eliminan 3 grupos de columnas antes de cualquier ingenieria:

**Leakage post-evento** — estas columnas solo existen DESPUES de que la transaccion fue procesada/rechazada. Usarlas entrenaria el modelo con informacion que no existiria en produccion al momento de predecir:
- `DE39_response_code`, `approved`, `response_description`, `DE44_additional_response_data`, `DE38_authorization_code`

**Identificadores y trazas** — cardinalidad >90%, sin poder predictivo, solo sirven para lookup interno:
- `transaction_id`, `pan_hash`, `DE2_PAN`, `DE11_STAN`, `DE37_retrieval_reference_number`, `DE35_track2_data_masked`, `DE56_original_data`, `DE58_authorizing_agent_id`, `DE100_receiving_institution_id`

**Columnas constantes o redundantes** — `MTI` es 100 para todas las filas (sin varianza). `bank_country` y `bank_code` son redundantes con el dataset mismo (Bolivia siempre es Bolivia). `DE50_currency_code_settlement`, `DE51_currency_code_billing`, `DE9_conversion_rate_billing` son redundantes con `currency_tx_alpha` y `amount_usd`. `DE7_transmission_datetime` y `DE13_local_date` + `DE12_local_time` se solapan con `hour_local` y `day_of_week` que ya estan derivados.

In [2]:
COLS_TO_DROP = [
    # Leakage post-evento
    'DE39_response_code', 'approved', 'response_description',
    'DE44_additional_response_data', 'DE38_authorization_code',

    # Identificadores y trazas
    'transaction_id', 'pan_hash', 'DE2_PAN', 'DE11_STAN',
    'DE37_retrieval_reference_number', 'DE35_track2_data_masked',
    'DE56_original_data', 'DE58_authorizing_agent_id',
    'DE100_receiving_institution_id',

    # Constantes o redundantes
    'MTI', 'bank_country', 'bank_code',
    'DE50_currency_code_settlement', 'DE51_currency_code_billing',
    'DE9_conversion_rate_billing', 'DE7_transmission_datetime',
    'DE13_local_date', 'DE12_local_time',
]

for name, df in [('bolivia', bolivia), ('brasil', brasil), ('guatemala', guatemala)]:
    cols_present = [c for c in COLS_TO_DROP if c in df.columns]
    cols_missing  = [c for c in COLS_TO_DROP if c not in df.columns]
    df.drop(columns=cols_present, inplace=True)
    print(f'{name}: eliminadas {len(cols_present)} columnas | shape resultante: {df.shape}')
    if cols_missing:
        print(f'  AVISO — no encontradas (ya no existian): {cols_missing}')

print('\nColumnas iguales en los 3 datasets:', list(bolivia.columns) == list(brasil.columns) == list(guatemala.columns))

bolivia: eliminadas 23 columnas | shape resultante: (100003, 43)
brasil: eliminadas 23 columnas | shape resultante: (100000, 43)
guatemala: eliminadas 23 columnas | shape resultante: (100000, 43)

Columnas iguales en los 3 datasets: True


## Ingenieria de Variables Nuevas

Se construyen 6 variables nuevas orientadas especificamente al objetivo de reducir falsos positivos en transacciones presenciales. Cada una se aplica de forma identica en los tres datasets.

1. **`log_amount_usd`** — log1p del monto en USD. La distribucion de montos tiene skewness severa; esta transformacion la normaliza y mejora la capacidad del modelo de separar montos anomalos sin que valores extremos dominen el entrenamiento.

2. **`amount_to_baseline_ratio`** — cociente entre `amount_usd` y `client_baseline_amount`. Captura si el monto de la transaccion es inusualmente alto respecto al patron historico del cliente. Un ratio >> 1 es señal de anomalia independientemente del monto absoluto.

3. **`is_chip_and_pin`** — flag binario: EMV presente (`DE55_emv_data_present = Y`) Y PIN usado (`DE52_pin_data_present = Y`). Representa el nivel mas alto de autenticacion en transacciones presenciales. Una transaccion con chip+pin tiene probabilidad de fraude significativamente menor y ayuda al modelo a no disparar falsas alertas en ese segmento.

4. **`is_high_risk_channel`** — flag binario: canal ECOM o MOTO (Card Not Present). Separa explicitamente los canales donde el fraude es estructuralmente mas alto (~8%) de los presenciales (~2.3%), evitando que el modelo contamine su aprendizaje entre ambos segmentos.

5. **`is_suspicious_pos_mode`** — flag binario: `DE22_pos_entry_mode` en valores 22 (fallback chip a banda, altamente sospechoso) o 81 (e-commerce). El modo 22 tiene 98% de tasa de fraude en el EDA — es la señal individual mas fuerte encontrada.

6. **`mcc_fraud_rate`** — target encoding del MCC: tasa de fraude historica por codigo de comercio, calculada SOLO sobre Bolivia + Brasil (train). Se propaga luego a Guatemala sin recalcular. Esto evita leakage y captura que ciertos rubros (MCC 6051, 7995) concentran casi todo el fraude.

In [3]:
# 1. log_amount_usd
for df in [bolivia, brasil, guatemala]:
    df['log_amount_usd'] = np.log1p(df['amount_usd'])

# 2. amount_to_baseline_ratio
for df in [bolivia, brasil, guatemala]:
    df['amount_to_baseline_ratio'] = df['amount_usd'] / df['client_baseline_amount'].replace(0, np.nan)
    df['amount_to_baseline_ratio'] = df['amount_to_baseline_ratio'].fillna(0)

# 3. is_chip_and_pin
for df in [bolivia, brasil, guatemala]:
    df['is_chip_and_pin'] = (
        (df['DE55_emv_data_present'] == 'Y') &
        (df['DE52_pin_data_present'] == 'Y')
    ).astype(int)

# 4. is_high_risk_channel
for df in [bolivia, brasil, guatemala]:
    df['is_high_risk_channel'] = df['channel'].isin(['ECOM', 'MOTO']).astype(int)

# 5. is_suspicious_pos_mode
for df in [bolivia, brasil, guatemala]:
    df['is_suspicious_pos_mode'] = df['DE22_pos_entry_mode'].isin([22, 81]).astype(int)

# 6. mcc_fraud_rate — calculado SOLO en train (bolivia + brasil), propagado a guatemala
train_combined = pd.concat([bolivia, brasil], ignore_index=True)
mcc_fraud_map = (
    train_combined.groupby('DE18_merchant_category_code')['is_fraud']
    .mean()
    .to_dict()
)
global_fraud_rate = train_combined['is_fraud'].mean()

for df in [bolivia, brasil, guatemala]:
    df['mcc_fraud_rate'] = (
        df['DE18_merchant_category_code']
        .map(mcc_fraud_map)
        .fillna(global_fraud_rate)
    )

# Verificacion
print('Variables nuevas creadas:')
new_cols = ['log_amount_usd', 'amount_to_baseline_ratio', 'is_chip_and_pin',
            'is_high_risk_channel', 'is_suspicious_pos_mode', 'mcc_fraud_rate']
for name, df in [('bolivia', bolivia), ('brasil', brasil), ('guatemala', guatemala)]:
    print(f'\n{name}:')
    print(df[new_cols].describe().round(4))

Variables nuevas creadas:

bolivia:
       log_amount_usd  amount_to_baseline_ratio  is_chip_and_pin  \
count     100003.0000               100003.0000      100003.0000   
mean           5.2758                    0.2750           0.3655   
std            1.3672                    0.3495           0.4816   
min            0.0677                    0.0000           0.0000   
25%            4.4456                    0.0601           0.0000   
50%            5.3054                    0.1349           0.0000   
75%            6.1840                    0.3147           1.0000   
max            8.8148                    3.4495           1.0000   

       is_high_risk_channel  is_suspicious_pos_mode  mcc_fraud_rate  
count           100003.0000             100003.0000     100003.0000  
mean                 0.5115                  0.4650          0.0440  
std                  0.4999                  0.4988          0.0842  
min                  0.0000                  0.0000          0.0068  
2

## Eliminacion de columnas restantes de bajo valor para el modelo

Luego de crear las variables nuevas, se eliminan columnas que no seran utiles como features directos en LightGBM:

**Columnas de texto libre o alta cardinalidad no encodeable de forma segura:**
- `DE43_card_acceptor_name_location`: nombre y ciudad del comercio en texto libre, demasiado especifico y no generalizable entre paises.
- `client_home_city`, `DE42_card_acceptor_id`, `DE41_terminal_id`: identificadores de alta cardinalidad sin patron explotable de forma directa.
- `pan_masked`: version enmascarada del PAN, no aporta nada que `card_brand` no cubra.

**Columnas ya absorbidas por variables nuevas:**
- `DE52_pin_data_present`, `DE55_emv_data_present`: ya capturadas en `is_chip_and_pin`.
- `channel`: ya capturada en `is_high_risk_channel`.
- `DE18_merchant_category_code`: ya capturada en `mcc_fraud_rate`. Se mantiene como categorica solo si se decide encodear, pero para LightGBM el target encoding es suficiente y mas robusto.

**Columnas con informacion duplicada o de muy bajo aporte:**
- `DE3_processing_code`, `DE23_card_seq_number`, `DE48_additional_data`, `DE54_additional_amounts`, `DE60_pos_terminal_type`, `DE61_pos_extended_data`, `DE63_network_specific`, `DE102_account_id_1`, `DE103_account_id_2`, `DE123_pos_data_code`: campos ISO 8583 crudos que o bien son constantes, tienen altisima cardinalidad, o su informacion util ya esta representada por otras variables.

In [4]:
COLS_TO_DROP_2 = [
    # Texto libre o alta cardinalidad
    'DE43_card_acceptor_name_location', 'client_home_city',
    'DE42_card_acceptor_id', 'DE41_terminal_id', 'pan_masked',

    # Absorbidas por variables nuevas
    'DE52_pin_data_present', 'DE55_emv_data_present',
    'channel', 'DE18_merchant_category_code',

    # Bajo aporte o duplicadas
    'DE3_processing_code', 'DE23_card_seq_number', 'DE48_additional_data',
    'DE54_additional_amounts', 'DE60_pos_terminal_type', 'DE61_pos_extended_data',
    'DE63_network_specific', 'DE102_account_id_1', 'DE103_account_id_2',
    'DE123_pos_data_code',

    'bank_name', 'bank_tier', 'client_segment', 'currency_tx_alpha'
]

for name, df in [('bolivia', bolivia), ('brasil', brasil), ('guatemala', guatemala)]:
    cols_present = [c for c in COLS_TO_DROP_2 if c in df.columns]
    cols_missing  = [c for c in COLS_TO_DROP_2 if c not in df.columns]
    df.drop(columns=cols_present, inplace=True)
    print(f'{name}: eliminadas {len(cols_present)} columnas | shape resultante: {df.shape}')
    if cols_missing:
        print(f'  AVISO — no encontradas: {cols_missing}')

print('\nColumnas iguales en los 3 datasets:', list(bolivia.columns) == list(brasil.columns) == list(guatemala.columns))
print('\nColumnas finales:')
print(bolivia.columns.tolist())

bolivia: eliminadas 23 columnas | shape resultante: (100003, 26)
brasil: eliminadas 23 columnas | shape resultante: (100000, 26)
guatemala: eliminadas 23 columnas | shape resultante: (100000, 26)

Columnas iguales en los 3 datasets: True

Columnas finales:
['client_id', 'card_brand', 'DE4_amount_transaction', 'DE6_amount_cardholder_billing', 'DE14_expiration_date', 'DE15_settlement_date', 'DE19_acquirer_country_code', 'DE22_pos_entry_mode', 'DE25_pos_condition_code', 'DE32_acquiring_institution_id', 'DE49_currency_code_transaction', 'amount_local', 'amount_tx_currency', 'amount_usd', 'is_international', 'distance_from_home_km', 'hour_local', 'day_of_week', 'client_baseline_amount', 'is_fraud', 'log_amount_usd', 'amount_to_baseline_ratio', 'is_chip_and_pin', 'is_high_risk_channel', 'is_suspicious_pos_mode', 'mcc_fraud_rate']


In [5]:
print('=== Verificacion de estructura final ===\n')

for name, df in [('bolivia', bolivia), ('brasil', brasil), ('guatemala', guatemala)]:
    print(f'{name}:')
    print(f'  Shape        : {df.shape}')
    print(f'  Columnas     : {len(df.columns)}')
    print(f'  Nulos totales: {df.isnull().sum().sum()}')
    print(f'  is_fraud     : {df["is_fraud"].sum()} positivos ({df["is_fraud"].mean()*100:.2f}%)')
    print()

print('Columnas identicas en los 3 datasets:', list(bolivia.columns) == list(brasil.columns) == list(guatemala.columns))

=== Verificacion de estructura final ===

bolivia:
  Shape        : (100003, 26)
  Columnas     : 26
  Nulos totales: 4007
  is_fraud     : 4919 positivos (4.92%)

brasil:
  Shape        : (100000, 26)
  Columnas     : 26
  Nulos totales: 4036
  is_fraud     : 3205 positivos (3.21%)

guatemala:
  Shape        : (100000, 26)
  Columnas     : 26
  Nulos totales: 111936
  is_fraud     : 0.0 positivos (nan%)

Columnas identicas en los 3 datasets: True


In [6]:
print('=== Nulos por columna ===\n')

null_summary = pd.DataFrame({
    'bolivia':   bolivia.isnull().sum(),
    'brasil':    brasil.isnull().sum(),
    'guatemala': guatemala.isnull().sum(),
})
null_summary['bolivia_pct']   = (null_summary['bolivia']   / len(bolivia)   * 100).round(2)
null_summary['brasil_pct']    = (null_summary['brasil']    / len(brasil)    * 100).round(2)
null_summary['guatemala_pct'] = (null_summary['guatemala'] / len(guatemala) * 100).round(2)

null_summary = null_summary[null_summary[['bolivia','brasil','guatemala']].sum(axis=1) > 0]
print(null_summary.sort_values('guatemala', ascending=False).to_string())

=== Nulos por columna ===

                       bolivia  brasil  guatemala  bolivia_pct  brasil_pct  guatemala_pct
is_fraud                     0       0     100000         0.00        0.00         100.00
day_of_week               1005    1024       3015         1.00        1.02           3.02
card_brand                1042    1010       3003         1.04        1.01           3.00
distance_from_home_km      978     977       2983         0.98        0.98           2.98
DE15_settlement_date       982    1025       2935         0.98        1.03           2.94


## Imputacion de nulos

Los nulos encontrados son de dos tipos:

**`is_fraud` en Guatemala** — no se imputa. Es el target a predecir, debe permanecer ausente.

**Columnas con ~1-3% de nulos** (`day_of_week`, `card_brand`, `distance_from_home_km`, `client_segment`, `DE15_settlement_date`):
- `distance_from_home_km`: variable numerica, se imputa con la mediana por ser robusta a outliers.
- `DE15_settlement_date`: variable numerica/codigo, se imputa con la mediana.
- `day_of_week`, `card_brand`, `client_segment`: variables categoricas, se imputa con la moda de cada dataset por separado para no contaminar distribuciones entre paises.

Todas las imputaciones se calculan sobre cada dataset de forma independiente para no filtrar informacion entre Bolivia, Brasil y Guatemala.

In [7]:
NUMERIC_IMPUTE  = ['distance_from_home_km', 'DE15_settlement_date']
CATEGORY_IMPUTE = ['day_of_week', 'card_brand', 'client_segment']

for name, df in [('bolivia', bolivia), ('brasil', brasil), ('guatemala', guatemala)]:
    for col in NUMERIC_IMPUTE:
        if col in df.columns:
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)

    for col in CATEGORY_IMPUTE:
        if col in df.columns:
            mode_val = df[col].mode()[0]
            df[col].fillna(mode_val, inplace=True)

print('=== Nulos restantes por dataset ===\n')
for name, df in [('bolivia', bolivia), ('brasil', brasil), ('guatemala', guatemala)]:
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0]
    print(f'{name}:')
    if len(nulos) == 0:
        print('  Ninguno')
    else:
        print(nulos.to_string())
    print()

=== Nulos restantes por dataset ===

bolivia:
  Ninguno

brasil:
  Ninguno

guatemala:
is_fraud    100000



## Exportacion de datasets finales

Se exportan los tres datasets con la ingenieria de variables aplicada en formato Parquet:

- `bolivia_fe.parquet` y `brasil_fe.parquet`: datasets de entrenamiento listos para LightGBM, con `is_fraud` conocido.
- `guatemala_fe.parquet`: dataset de test con `is_fraud` en NaN — es la variable que el modelo debe predecir.

In [8]:
import os

train_dir = '../parquet/train/'
test_dir  = '../parquet/test/'

os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir,  exist_ok=True)

bolivia.to_parquet(f'{train_dir}bolivia_B.parquet', index=False)
brasil.to_parquet(f'{train_dir}brasil_B.parquet',   index=False)
guatemala.to_parquet(f'{test_dir}guatemala_B.parquet', index=False)

print('Archivos exportados:')
for path, df in [
    (f'{train_dir}bolivia_B.parquet',   bolivia),
    (f'{train_dir}brasil_B.parquet',    brasil),
    (f'{test_dir}guatemala_B.parquet',  guatemala),
]:
    size_mb = os.path.getsize(path) / 1024 / 1024
    print(f'  {path}: {df.shape} | {size_mb:.2f} MB')

Archivos exportados:
  ../parquet/train/bolivia_B.parquet: (100003, 26) | 5.51 MB
  ../parquet/train/brasil_B.parquet: (100000, 26) | 4.73 MB
  ../parquet/test/guatemala_B.parquet: (100000, 26) | 3.72 MB
